In [1]:
import os
print(os.getcwd())   # xem kernel đang đứng ở đâu


d:\ct551_v2


In [6]:
import re
from datetime import datetime
import numpy as np
import pandas as pd

path_dataset = "dataset_high/HI-Small_Patterns.txt"   # fix: thiếu chữ "s"

def parse_pattern(path):
    pattern_list = []
    pattern_type = None
    pattern_ts = []
    with open(path) as f:
        for raw in f:
            line = raw.rstrip("\n")
            if not line.strip():
                continue
            if line.startswith("BEGIN LAUNDERING ATTEMPT"):      # fix: thụt vào trong for-loop
                rest = line.split("-", 1)[1].strip()
                pattern_type = rest.split(":")[0].strip()
                pattern_ts = []
            elif line.startswith("END LAUNDERING ATTEMPT"):
                pattern_list.append((pattern_type, pattern_ts))
                pattern_type = None
                pattern_ts = []
            else:
                ts_str = line.split(",", 1)[0]
                pattern_ts.append(datetime.strptime(ts_str, "%Y/%m/%d %H:%M"))  # fix: strptime
        # fix: return đặt sau khi vòng for chạy hết, không nằm trong nhánh else
    return pattern_list

def summarize(pattern_list):
    rows = []
    for pattern_type, ts_list in pattern_list:
        n_tx = len(ts_list)
        span_hour = (max(ts_list) - min(ts_list)).total_seconds() / 3600
        rows.append({"pattern": pattern_type, "n_tx": n_tx, "span_h": span_hour})
    df = pd.DataFrame(rows)
    summary = df.groupby("pattern").agg(
        n_transaction=("n_tx", "size"),
        median_tx=("n_tx", "median"),
        median_span=("span_h", "median"),                          # fix: median thật
        p90_span=("span_h", lambda s: np.percentile(s, 90)),        # fix: tách riêng p90
    ).round(2)
    return summary.sort_values("median_span")

if __name__ == "__main__":
    pattern_list = parse_pattern(path_dataset)          # fix: gọi parse thay vì hardcode []
    summary = summarize(pattern_list)
    print(summary)
    print(f"total_attempts: {len(pattern_list)}")       # fix: thêm f-string + biến đúng


                n_transaction  median_tx  median_span  p90_span
pattern                                                        
BIPARTITE                  49        4.0        24.33     43.12
RANDOM                     41        3.0        46.00     86.63
CYCLE                      54        4.0        71.88     90.35
STACK                      43       10.0        73.32    101.00
FAN-OUT                    48        7.0        76.66     94.71
FAN-IN                     40        8.0        84.93     94.70
SCATTER-GATHER             44       14.0        88.77     95.50
GATHER-SCATTER             51       14.0       150.82    183.83
total_attempts: 370
